1 - load dataset

In [33]:
import pandas as pd

In [34]:
df = pd.read_csv('final_combined_df.csv')

In [35]:
df.head()

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,s2553,s2554,s2555,s2556,s2557,s2558,s2559,s2560,gender,height_cm
0,-0.053796,-0.082064,-0.167094,0.602376,0.293155,-0.111758,-0.173058,-0.189721,-0.074341,0.21857,...,-0.151895,-0.155769,-0.139172,-0.239405,-0.013789,-0.197436,-0.148977,0.17587,0,175


2 - load models and scalers

In [36]:
!pip install scikit-learn


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [37]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # disable GPU

In [38]:
import os
import pickle
from tensorflow.keras.models import load_model

folder = "downloaded_artifacts"

# 1) Load model
model_path = os.path.join(folder, "eff_ann_version8.h5")
model = load_model(model_path, compile=False)

# 2) Load scalers
robust_features_path = os.path.join(folder, "scaler_robust_features.pkl")
standard_features_path = os.path.join(folder, "scaler_standard_features.pkl")
targets_path = os.path.join(folder, "scaler_targets.pkl")

with open(robust_features_path, "rb") as f:
    scaler_robust_features = pickle.load(f)

with open(standard_features_path, "rb") as f:
    scaler_standard_features = pickle.load(f)

with open(targets_path, "rb") as f:
    scaler_targets = pickle.load(f)

print("Model loaded:", type(model))
print("Robust feature scaler loaded:", type(scaler_robust_features))
print("Standard feature scaler loaded:", type(scaler_standard_features))
print("Target scaler loaded:", type(scaler_targets))

Model loaded: <class 'keras.src.models.sequential.Sequential'>
Robust feature scaler loaded: <class 'sklearn.preprocessing._data.RobustScaler'>
Standard feature scaler loaded: <class 'sklearn.preprocessing._data.StandardScaler'>
Target scaler loaded: <class 'sklearn.preprocessing._data.StandardScaler'>


/tmp/ipykernel_4550/823403589.py:17: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  scaler_robust_features = pickle.load(f)
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/t

3 - scaling

In [39]:
import pandas as pd

# Copy original dataframe
df_scaled = df.copy()

# Columns
no_scale_cols = ["gender"]
standard_cols = ["height_cm"]

# Robust scaler columns = everything except gender & height_cm
robust_cols = [col for col in df.columns if col not in no_scale_cols + standard_cols]

# --- Apply scaling ---

# 1) Robust scaling
df_scaled[robust_cols] = scaler_robust_features.transform(df[robust_cols])

# 2) Standard scaling for height
df_scaled[standard_cols] = scaler_standard_features.transform(df[standard_cols])

# gender remains unchanged

# --- Result ---
print(df_scaled.head())

         f1        f2        f3        f4        f5        f6        f7  \
0 -1.747153 -0.960525 -1.073681  0.036224  0.337957 -0.412064 -1.557985   

         f8        f9       f10  ...     s2553    s2554     s2555     s2556  \
0 -2.413018 -1.819649 -0.032738  ... -0.091332 -1.50779  0.052008 -0.983502   

      s2557    s2558     s2559     s2560  gender  height_cm  
0  1.338983  1.37941  0.566999 -0.634578       0   0.381995  

[1 rows x 5122 columns]


4 - predict the target features

In [40]:
import pandas as pd

# define targets
target_cols = [
    'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip',
    'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh',
    'waist', 'wrist', 'weight_kg'
]

# 1) predict (scaled)
y_pred_scaled = model.predict(df_scaled, verbose=0)

# 2) to dataframe
y_pred_scaled_df = pd.DataFrame(
    y_pred_scaled,
    columns=target_cols,
    index=df_scaled.index
)

# 3) inverse scale
y_pred = scaler_targets.inverse_transform(y_pred_scaled_df)

# 4) final df
y_pred_df = pd.DataFrame(
    y_pred,
    columns=target_cols,
    index=df_scaled.index
)

print(y_pred_df.head())

       ankle  arm-length      bicep       calf      chest    forearm  \
0  25.991245   50.256401  35.509655  40.378853  114.67482  27.671104   

          hip  leg-length  shoulder-breadth  shoulder-to-crotch      thigh  \
0  113.373825   76.850197         39.724091           72.802361  59.478497   

        waist      wrist  weight_kg  
0  106.165588  16.889267  92.109314  
